In [3]:
import os 

os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

from transformers import AutoConfig, FlaxAutoModel, AutoTokenizer, FlaxBertPreTrainedModel
from transformers.models.bert.modeling_flax_bert import FlaxBertModule
import jax
from jax import numpy as jnp
from flax import linen as nn
from flax.struct import PyTreeNode
from flax import nnx
from flax.nnx import bridge

# Download model and configuration from huggingface.co and cache.
bert_model: FlaxBertPreTrainedModel = FlaxAutoModel.from_pretrained('facebook/contriever', revision="main", from_pt=True)
tokenizer = AutoTokenizer.from_pretrained('facebook/contriever', use_fast=True, revision="main", clean_up_tokenization_spaces=True)

Some weights of the model checkpoint at facebook/contriever were not used when initializing FlaxBertModel: {('embeddings', 'position_ids')}
- This IS expected if you are initializing FlaxBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing FlaxBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [4]:
import sys
import os

repo_dir = os.path.dirname(os.path.abspath("./"))
if repo_dir not in sys.path:
    print(f'add repository dir: {repo_dir}')
    sys.path.append(repo_dir)

add repository dir: /home/nazar/projects/multi-step-retrieval-rl


In [5]:
sentences = [
    "Where was Marie Curie born?",
    "Maria Sklodowska, later known as Marie Curie, was born on November 7, 1867.",
    "Born in Paris on 15 May 1859, Pierre Curie was the son of Eugène Curie, a doctor of French Catholic origin from Alsace."
] 

# Apply tokenizer
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='jax')




In [6]:

from babilong_fix import QA2FixWrapper
from rl.retrieval_babilong import RetrNoiseInjectionDataset, RetrSentenceSampler
from babilong_utils import TaskDataset
from datasets import Dataset, load_dataset, load_from_disk
import datasets
import sys
import time
import numpy as np
from collections import deque
from rl.babilong_env import BabilongEnv
from rl.jax_text_env import TextReplayBuffer
from tqdm import tqdm


# torch.set_default_device('cuda:1')

max_steps = 10
num_sentences = 50
task = "qa2_two-supporting-facts"
noise_train_path = "/home/nazar/pg19-train-with-sentences"
noise_path_test = "/home/nazar/pg19-test-with-sentences"
facts_train_path = f"/home/nazar/tasks_1-20_v1-2/en-10k/{task}_train.txt"
facts_test_path = f"/home/nazar/tasks_1-20_v1-2/en-10k/{task}_test.txt"


fact_dataset = QA2FixWrapper(TaskDataset(facts_train_path), add_sentence_idx=True)  
test_fact_dataset = QA2FixWrapper(TaskDataset(facts_test_path), add_sentence_idx=True)   

noise_dataset = datasets.load_from_disk(noise_path_test)
noise_sampler = RetrSentenceSampler(noise_dataset)

dataset = RetrNoiseInjectionDataset(
    task_dataset=fact_dataset,
    noise_sentence_sampler=noise_sampler,
    num_sentences=num_sentences
)

dataset_test = RetrNoiseInjectionDataset(
    task_dataset=test_fact_dataset,
    noise_sentence_sampler=noise_sampler,
    num_sentences=num_sentences
)


In [8]:
from rl.flax_dqn import FlaxDQN, DQNArgs

agent = FlaxDQN(
    bert_model, 
    DQNArgs(gamma=0.99, tau=0.01,  lr=5e-5, max_steps=(20_000 // 4) * max_steps)
)

nnx.update(agent.action_embed_target.action_embed, nnx.state(agent.action_embed_target.action_embed) )

env = BabilongEnv( 
    embedder=agent.action_embed_target,
    embed_tokenizer=tokenizer, 
    dataset=dataset,
    max_steps=max_steps,
    max_embed_length=64
)

k = 0

env.reset()
step = 0
learning_start = 2_000
R = 0

buffer = TextReplayBuffer(50_000, tokenizer = tokenizer)

for ep_number in tqdm(range(20_000)):

    s = env.reset()
    a_embeds = env.get_extra_embeds(agent.action_embed)

    qf_loss, alpha_loss = 0, 0
    r_sum = 0
    done = False

    a_all = []

    while not done:
        step += 1
        
        action = agent.select_action(s, a_embeds, random = step < learning_start)
        s_next, a_data, reward, done = env.step(action)
        buffer.add(s, a_data, s_next, reward, done, 0)
        
        s = s_next
        R += reward
        r_sum += reward
        a_all.append(action)
        
        if step > learning_start and step % 4 == 0:
            s_batch, a_batch, next_s_batch, r_batch, not_done_batch, entropy_batch = buffer.sample(16)
            qf_loss = agent.update(s_batch, a_batch, next_s_batch, r_batch, not_done_batch)
        
    
    
   

  1%|▏         | 254/20000 [02:33<2:03:42,  2.66it/s] 

In [10]:
s0 = env.reset()
s1  = env.step(2)

In [11]:
s1

(TextMemory(item_ids=[2], available_ids={0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49}, available_mask=array([ True,  True, False,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True]), input_ids=array([  101,  2073,  2003,  1996,  6501,  1029,   102,   102,   101,
         1996,  2332,  2453,  5262,  2022,  5391,  2000,  3277,  1996,
        25697,  2015,  2005,  2049, 24814,  1025,  2145,  2009,  2001,
         2013,  1996,  2332,  1521,  1055, 25697,  2008,  1996,  3323,
         2941,  5173,  2049,  2108,  1998,  2049,  4204,  1012, 

In [7]:
from nnx_train_state import TS
import optax


q_model = QModule(bert_model)
q_model.train() # set deterministic=False


class TrainState(nnx.TrainState):
    other_variables: nnx.State

# state: TS[QModule] = TS.create(
#   model=q_model,
#   tx=optax.adam(1e-3)
# )
graph, model_params, model_stats = nnx.split(q_model, nnx.Param, ...)
state = TrainState.create(graph, params=model_params, other_variables=model_stats, tx=optax.adam(1e-3))


In [7]:
# @jax.jit
def train_step(state: TrainState, s_ids, s_att):

  model = nnx.merge(state.graphdef, state.params, state.other_variables)

  def loss_fn(m):
    res1 = m({"input_ids": s_ids, "attention_mask": s_att}, {"input_ids": s_ids, "attention_mask": s_att})
    return (res1 ** 2).mean()
  
  loss, grads = nnx.value_and_grad(loss_fn)(model)
  _, _, model_stats = nnx.split(model, nnx.Param, ...)

  state = state.apply_gradients(grads=grads, other_variables=model_stats)

  return state, loss

inputs = env.reset().embeds

jax_train = jax.jit(train_step).lower(state, inputs["input_ids"], inputs["attention_mask"]).compile()

In [5]:
# @jax.jit
# def train_step(state: TS[QModule], s_ids, s_att):

#   model = state.get_model()

#   def loss_fn(m):
#     res1 = m({"input_ids": s_ids, "attention_mask": s_att}, {"input_ids": s_ids, "attention_mask": s_att})
#     return (res1 ** 2).mean()
  
#   loss, grads  = nnx.value_and_grad(loss_fn)(model)
#   state = state.apply_gradients(model, grads=grads)

#   return state, loss


# @nnx.jit  # Automatic state management
# def train_step(model, opt_state, s_ids, s_att):

#   # model = nnx.merge(graphdef, state)
  
#   def loss_fn(m):
#     res = m({"input_ids": s_ids, "attention_mask": s_att}, {"input_ids": s_ids, "attention_mask": s_att})
#     return (res ** 2).mean()

#   loss, grads = nnx.value_and_grad(loss_fn)(model)
#   # optimizer.update(grads) 

#   params = nnx.state(model, nnx.Param)
#   updates, opt_state = opt2.update(grads, opt_state)
#   params = optax.apply_updates(params, updates)

#   nnx.update(model, params)

#   # graphdef, state = nnx.split(model)

#   return loss, model, opt_state
  

def fast_apply(model, s_ids, s_att):
    return model({"input_ids": s_ids, "attention_mask": s_att}, inputs)


jax_apply = nnx.jit(fast_apply)

In [9]:
from tqdm.notebook import tqdm


for i in tqdm(range(1000)):
    # state, loss = jax_train(state, inputs["input_ids"], inputs["attention_mask"])
    # graphdef, state = train_step(graphdef, state, inputs["input_ids"], inputs["attention_mask"]) 
    # loss = 0
    inputs = env.reset().embeds
    print(inputs["input_ids"].shape)
    embeds = jax_apply(q_model, inputs["input_ids"], inputs["attention_mask"])
    if i % 100 == 0:
        print(state.step)


  0%|          | 0/1000 [00:00<?, ?it/s]

(50, 88)
0
(50, 111)
(50, 66)
(50, 62)
(50, 87)
(50, 46)
(50, 65)
(50, 43)
(50, 39)
(50, 72)
(50, 41)
(50, 126)
(50, 51)
(50, 42)
(50, 85)
(50, 62)
(50, 100)
(50, 83)
(50, 55)


KeyboardInterrupt: 